In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import os
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("All libraries loaded successfully")

All libraries loaded successfully


In [2]:
# Set this to wherever your Marketing_Channel_Project folder is
DATA_PATH = r"C:\Users\Mayank Joshi\Downloads\Marketing_Channel_Project\data"

# Verify all files are present
expected_files = [
    'olist_marketing_qualified_leads_dataset.csv',
    'olist_closed_deals_dataset.csv',
    'olist_orders_dataset.csv',
    'olist_order_items_dataset.csv',
    'olist_order_payments_dataset.csv',
    'olist_order_reviews_dataset.csv',
    'olist_customers_dataset.csv',
    'olist_sellers_dataset.csv',
    'olist_products_dataset.csv',
    'olist_geolocation_dataset.csv',
    'product_category_name_translation.csv'
]

print("Checking files...\n")
all_found = True
for f in expected_files:
    full_path = os.path.join(DATA_PATH, f)
    exists = os.path.exists(full_path)
    try:
        size_kb = round(os.path.getsize(full_path) / 1024) if exists else 0
    except OSError:
        size_kb = 0
    status = "FOUND" if exists else "MISSING"
    print(f"  {status}  {f}  ({size_kb:,} KB)")
    if not exists:
        all_found = False

print(f"\n{'All files found — ready to load!' if all_found else 'Some files are missing — check your DATA_PATH'}")

Checking files...

  FOUND  olist_marketing_qualified_leads_dataset.csv  (687 KB)
  FOUND  olist_closed_deals_dataset.csv  (167 KB)
  FOUND  olist_orders_dataset.csv  (17,241 KB)
  FOUND  olist_order_items_dataset.csv  (15,077 KB)
  FOUND  olist_order_payments_dataset.csv  (5,642 KB)
  FOUND  olist_order_reviews_dataset.csv  (14,113 KB)
  FOUND  olist_customers_dataset.csv  (8,822 KB)
  FOUND  olist_sellers_dataset.csv  (171 KB)
  FOUND  olist_products_dataset.csv  (2,324 KB)
  FOUND  olist_geolocation_dataset.csv  (59,838 KB)
  FOUND  product_category_name_translation.csv  (3 KB)

All files found — ready to load!


In [3]:
print("Loading datasets...\n")

# Dataset 1 — Marketing Funnel
mql       = pd.read_csv(os.path.join(DATA_PATH, 'olist_marketing_qualified_leads_dataset.csv'))
deals     = pd.read_csv(os.path.join(DATA_PATH, 'olist_closed_deals_dataset.csv'))

# Dataset 2 — E-Commerce
orders    = pd.read_csv(os.path.join(DATA_PATH, 'olist_orders_dataset.csv'))
items     = pd.read_csv(os.path.join(DATA_PATH, 'olist_order_items_dataset.csv'))
payments  = pd.read_csv(os.path.join(DATA_PATH, 'olist_order_payments_dataset.csv'))
reviews   = pd.read_csv(os.path.join(DATA_PATH, 'olist_order_reviews_dataset.csv'))
customers = pd.read_csv(os.path.join(DATA_PATH, 'olist_customers_dataset.csv'))
sellers   = pd.read_csv(os.path.join(DATA_PATH, 'olist_sellers_dataset.csv'))
products  = pd.read_csv(os.path.join(DATA_PATH, 'olist_products_dataset.csv'))
geo       = pd.read_csv(os.path.join(DATA_PATH, 'olist_geolocation_dataset.csv'))
cat_trans = pd.read_csv(os.path.join(DATA_PATH, 'product_category_name_translation.csv'))

datasets = {
    'mql':       mql,
    'deals':     deals,
    'orders':    orders,
    'items':     items,
    'payments':  payments,
    'reviews':   reviews,
    'customers': customers,
    'sellers':   sellers,
    'products':  products,
    'geo':       geo,
    'cat_trans': cat_trans
}

print(f"{'Dataset':<15} {'Rows':>10} {'Columns':>10}")
print("-" * 38)
for name, df in datasets.items():
    print(f"{name:<15} {len(df):>10,} {len(df.columns):>10}")

Loading datasets...



Dataset               Rows    Columns
--------------------------------------
mql                  8,000          4
deals                  842         14
orders              99,441          8
items              112,650          7
payments           103,886          5
reviews             99,224          7
customers           99,441          5
sellers              3,095          4
products            32,951          9
geo              1,000,163          5
cat_trans               71          2


In [4]:
# Convert all date columns to proper datetime
date_cols = {
    'mql':    ['first_contact_date'],
    'deals':  ['won_date'],
    'orders': ['order_purchase_timestamp', 'order_approved_at',
               'order_delivered_carrier_date', 'order_delivered_customer_date',
               'order_estimated_delivery_date']
}

for df_name, cols in date_cols.items():
    df = datasets[df_name]
    for col in cols:
        df[col] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')
    print(f"{df_name}: converted {cols}")

# Re-assign cleaned versions
mql    = datasets['mql']
deals  = datasets['deals']
orders = datasets['orders']

print("\nDate ranges:")
print(f"  MQL first contact:   {mql['first_contact_date'].min().date()}  →  {mql['first_contact_date'].max().date()}")
print(f"  Deals won:           {deals['won_date'].min().date()}  →  {deals['won_date'].max().date()}")
print(f"  Orders purchased:    {orders['order_purchase_timestamp'].min().date()}  →  {orders['order_purchase_timestamp'].max().date()}")

mql: converted ['first_contact_date']
deals: converted ['won_date']


orders: converted ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Date ranges:
  MQL first contact:   2017-01-08  →  2018-12-05
  Deals won:           2017-12-05  →  2018-11-14
  Orders purchased:    2016-02-10  →  2018-12-09


In [5]:
# Merge Portuguese category names → English
products = products.merge(cat_trans, on='product_category_name', how='left')

# Check how many translated successfully
translated = products['product_category_name_english'].notna().sum()
total = len(products)
print(f"{translated:,} of {total:,} products have English category names ({round(translated/total*100, 1)}%)")
print(f"   {total - translated} products have no category translation (nulls or untranslated)")

# Preview
products[['product_id', 'product_category_name', 'product_category_name_english']].head(8)

32,328 of 32,951 products have English category names (98.1%)
   623 products have no category translation (nulls or untranslated)


,product_id,product_category_name,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares
5,41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,musical_instruments
6,732bd381ad09e530fe0a5f457d81becb,cool_stuff,cool_stuff
7,2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,furniture_decor


In [6]:
print("=" * 55)
print("NULL AUDIT — ALL DATASETS")
print("=" * 55)

for name, df in datasets.items():
    null_counts = df.isnull().sum()
    null_cols = null_counts[null_counts > 0]
    if len(null_cols) > 0:
        print(f"\n📋 {name.upper()}")
        for col, count in null_cols.items():
            pct = round(count / len(df) * 100, 1)
            bar = '█' * int(pct / 5)
            print(f"   {col:<40} {count:>6,}  ({pct:>5}%)  {bar}")
    else:
        print(f"\n{name.upper()} — no nulls")

NULL AUDIT — ALL DATASETS

📋 MQL
   first_contact_date                        4,942  ( 61.8%)  ████████████
   origin                                       60  (  0.8%)  

📋 DEALS
   business_segment                              1  (  0.1%)  
   lead_type                                     6  (  0.7%)  
   lead_behaviour_profile                      177  ( 21.0%)  ████
   has_company                                 779  ( 92.5%)  ██████████████████
   has_gtin                                    778  ( 92.4%)  ██████████████████
   average_stock                               776  ( 92.2%)  ██████████████████
   business_type                                10  (  1.2%)  
   declared_product_catalog_size               773  ( 91.8%)  ██████████████████

📋 ORDERS
   order_purchase_timestamp                 59,810  ( 60.1%)  ████████████
   order_approved_at                        59,793  ( 60.1%)  ████████████
   order_delivered_carrier_date             62,109  ( 62.5%)  ████████████
   or

In [7]:
print("=" * 55)
print("JOIN INTEGRITY CHECK")
print("=" * 55)

total_mqls       = len(mql)
converted_mqls   = mql['mql_id'].isin(deals['mql_id']).sum()
unconverted_mqls = total_mqls - converted_mqls
conversion_rate  = round(converted_mqls / total_mqls * 100, 2)

print(f"\n  Total MQLs (leads):          {total_mqls:>7,}")
print(f"  Converted to closed deals:   {converted_mqls:>7,}  ({conversion_rate}%)")
print(f"  Did not convert (dropped):   {unconverted_mqls:>7,}  ({round(100 - conversion_rate, 2)}%)")

# How many closed deal sellers exist in the sellers table
deals_with_seller = deals['seller_id'].isin(sellers['seller_id']).sum()
print(f"\n  Closed deals matched to sellers table: {deals_with_seller:,} of {len(deals):,}")

# How many sellers have order data
sellers_with_orders = sellers['seller_id'].isin(items['seller_id']).sum()
print(f"  Sellers with at least 1 order:         {sellers_with_orders:,} of {len(sellers):,}")

print(f"\nRaw funnel conversion rate: {conversion_rate}%")
print("   This is your baseline — we'll break this down by channel in Stage 2")

JOIN INTEGRITY CHECK

  Total MQLs (leads):            8,000
  Converted to closed deals:       842  (10.52%)
  Did not convert (dropped):     7,158  (89.48%)

  Closed deals matched to sellers table: 380 of 842
  Sellers with at least 1 order:         3,095 of 3,095

Raw funnel conversion rate: 10.52%
   This is your baseline — we'll break this down by channel in Stage 2


In [8]:
print("Building master table...\n")

# Step 1: MQL + deals (full funnel for every lead)
master = mql.merge(deals, on='mql_id', how='left')
master['converted'] = master['seller_id'].notna().astype(int)
print(f"Step 1 — MQL + deals:              {len(master):,} rows")

# Step 2: Add seller location
master = master.merge(
    sellers[['seller_id', 'seller_city', 'seller_state', 'seller_zip_code_prefix']],
    on='seller_id', how='left'
)
print(f"Step 2 — + seller info:            {len(master):,} rows")

# Step 3: Revenue per seller from order items
seller_revenue = (
    items.groupby('seller_id')
         .agg(
             total_revenue   = ('price', 'sum'),
             total_orders     = ('order_id', 'nunique'),
             total_items_sold = ('order_item_id', 'count'),
             avg_item_price   = ('price', 'mean')
         )
         .reset_index()
)
master = master.merge(seller_revenue, on='seller_id', how='left')
print(f"Step 3 — + revenue/order data:     {len(master):,} rows")

# Step 4: Average review score per seller
seller_reviews = (
    items[['order_id', 'seller_id']]
    .merge(reviews[['order_id', 'review_score']], on='order_id', how='left')
    .groupby('seller_id')['review_score']
    .mean()
    .reset_index()
    .rename(columns={'review_score': 'avg_review_score'})
)
master = master.merge(seller_reviews, on='seller_id', how='left')
print(f"Step 4 — + review scores:          {len(master):,} rows")

# Step 5: Days to convert
master['days_to_convert'] = (
    master['won_date'] - master['first_contact_date']
).dt.days
print(f"Step 5 — + days to convert:        {len(master):,} rows")

# Step 6: Month/year columns for time analysis
master['contact_month']  = master['first_contact_date'].dt.to_period('M')
master['won_month']      = master['won_date'].dt.to_period('M')

print(f"\nMaster table built!")
print(f"   Shape: {master.shape[0]:,} rows × {master.shape[1]} columns")
print(f"\nColumns: {list(master.columns)}")

Building master table...

Step 1 — MQL + deals:              8,000 rows
Step 2 — + seller info:            8,000 rows


Step 3 — + revenue/order data:     8,000 rows
Step 4 — + review scores:          8,000 rows
Step 5 — + days to convert:        8,000 rows

Master table built!
   Shape: 8,000 rows × 29 columns

Columns: ['mql_id', 'first_contact_date', 'landing_page_id', 'origin', 'seller_id', 'sdr_id', 'sr_id', 'won_date', 'business_segment', 'lead_type', 'lead_behaviour_profile', 'has_company', 'has_gtin', 'average_stock', 'business_type', 'declared_product_catalog_size', 'declared_monthly_revenue', 'converted', 'seller_city', 'seller_state', 'seller_zip_code_prefix', 'total_revenue', 'total_orders', 'total_items_sold', 'avg_item_price', 'avg_review_score', 'days_to_convert', 'contact_month', 'won_month']


In [9]:
print("MASTER TABLE SUMMARY")
print("=" * 55)

converted = master[master['converted'] == 1]
not_converted = master[master['converted'] == 0]

print(f"\n  Total leads in master table:   {len(master):,}")
print(f"  Converted sellers:             {len(converted):,}")
print(f"  Not converted:                 {len(not_converted):,}")
print(f"\n  Converted sellers with revenue data:")
print(f"    Avg total revenue per seller:  ${converted['total_revenue'].mean():,.2f}")
print(f"    Median revenue per seller:     ${converted['total_revenue'].median():,.2f}")
print(f"    Max revenue (single seller):   ${converted['total_revenue'].max():,.2f}")
print(f"\n  Avg days to convert:           {converted['days_to_convert'].mean():.0f} days")
print(f"  Avg review score:              {converted['avg_review_score'].mean():.2f} / 5.0")

print(f"\n  Channels present:")
for ch, count in master['origin'].value_counts().items():
    pct = round(count / len(master) * 100, 1)
    print(f"    {ch:<25} {count:>5,}  ({pct}%)")

# Save master table
master.to_csv(os.path.join(DATA_PATH, 'master_table.csv'), index=False)
print(f"\nmaster_table.csv saved to your project folder")

MASTER TABLE SUMMARY

  Total leads in master table:   8,000
  Converted sellers:             842
  Not converted:                 7,158

  Converted sellers with revenue data:
    Avg total revenue per seller:  $1,781.19
    Median revenue per seller:     $547.40
    Max revenue (single seller):   $113,628.97

  Avg days to convert:           -51 days
  Avg review score:              4.27 / 5.0

  Channels present:
    organic_search            2,296  (28.7%)
    paid_search               1,586  (19.8%)
    social                    1,350  (16.9%)
    unknown                   1,099  (13.7%)
    direct_traffic              499  (6.2%)
    email                       493  (6.2%)
    referral                    284  (3.5%)
    other                       150  (1.9%)
    display                     118  (1.5%)
    other_publicities            65  (0.8%)



master_table.csv saved to your project folder


### Data Quality Note — Missing first_contact_date

**Issue:** 4,942 of 8,000 MQL leads (61.8%) have no `first_contact_date` recorded.

**Root cause:** Leads imported from external sources or untagged campaigns 
were not assigned a contact date in the CRM system.

**How we handle it:**
- A `has_contact_date` boolean flag is added to the master table
- All time-series and cohort analysis filters to `has_contact_date == 1` only
- Days-to-convert is only calculated where both dates are present and positive
- This limits time-based analysis to **3,058 leads** (38.2% of total)

**Implication for analysis:** Conversion rate trends and cohort analysis 
reflect the trackable subset of leads, not the full 8,000. 
Overall conversion metrics (10.52%) use the full dataset.

*Tracked in GitHub Issue #1*

In [10]:
# Re-load orders with correct timestamp format
orders = pd.read_csv(os.path.join(DATA_PATH, 'olist_orders_dataset.csv'))

timestamp_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in timestamp_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

null_after = orders[timestamp_cols].isnull().sum()
print("Null counts after re-parsing:\n")
for col in timestamp_cols:
    print(f"  {col:<45} {null_after[col]:>5,}")

print(f"\nDate range check:")
print(f"  Earliest order: {orders['order_purchase_timestamp'].min()}")
print(f"  Latest order:   {orders['order_purchase_timestamp'].max()}")

Null counts after re-parsing:

  order_purchase_timestamp                          0
  order_approved_at                               160
  order_delivered_carrier_date                  1,783
  order_delivered_customer_date                 2,965
  order_estimated_delivery_date                     0

Date range check:
  Earliest order: 2016-09-04 21:15:19
  Latest order:   2018-10-17 17:30:18


In [11]:
# Fix lead_behaviour_profile nulls
deals['lead_behaviour_profile'] = deals['lead_behaviour_profile'].fillna('unclassified')

# Fix product category nulls
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')
products['product_category_name']         = products['product_category_name'].fillna('unknown')

# Flag MQL rows with missing contact date — do not drop, just mark them
mql['has_contact_date'] = mql['first_contact_date'].notna().astype(int)

# Fill small nulls
mql['origin']             = mql['origin'].fillna('unknown')
deals['business_segment'] = deals['business_segment'].fillna('unknown')
deals['business_type']    = deals['business_type'].fillna('unknown')
deals['lead_type']        = deals['lead_type'].fillna('unknown')

print("Null fixes applied:\n")
print(f"  lead_behaviour_profile nulls remaining : {deals['lead_behaviour_profile'].isnull().sum()}")
print(f"  product_category_name_english nulls    : {products['product_category_name_english'].isnull().sum()}")
print(f"  mql origin nulls                       : {mql['origin'].isnull().sum()}")
print(f"  MQL rows WITH contact date             : {mql['has_contact_date'].sum():,}")
print(f"  MQL rows WITHOUT contact date          : {(mql['has_contact_date']==0).sum():,}")
print(f"\nAll nulls handled — ready to rebuild master table")

Null fixes applied:

  lead_behaviour_profile nulls remaining : 0
  product_category_name_english nulls    : 0
  mql origin nulls                       : 0
  MQL rows WITH contact date             : 3,058
  MQL rows WITHOUT contact date          : 4,942

All nulls handled — ready to rebuild master table


In [12]:
print("Rebuilding master table with clean data...\n")

# Step 1: MQL + deals
master = mql.merge(deals, on='mql_id', how='left')
master['converted'] = master['seller_id'].notna().astype(int)

# Step 2: Seller location
master = master.merge(
    sellers[['seller_id', 'seller_city', 'seller_state', 'seller_zip_code_prefix']],
    on='seller_id', how='left'
)

# Step 3: Revenue per seller from order items
seller_revenue = (
    items.groupby('seller_id')
         .agg(
             total_revenue    = ('price', 'sum'),
             total_orders     = ('order_id', 'nunique'),
             total_items_sold = ('order_item_id', 'count'),
             avg_item_price   = ('price', 'mean')
         )
         .reset_index()
)
master = master.merge(seller_revenue, on='seller_id', how='left')

# Step 4: Avg review score per seller
seller_reviews = (
    items[['order_id', 'seller_id']]
    .merge(reviews[['order_id', 'review_score']], on='order_id', how='left')
    .groupby('seller_id')['review_score']
    .mean()
    .reset_index()
    .rename(columns={'review_score': 'avg_review_score'})
)
master = master.merge(seller_reviews, on='seller_id', how='left')

# Step 5: Fix days to convert using corrected date parsing
mql_dates  = mql[['mql_id', 'first_contact_date']].copy()
deals_dates = deals[['mql_id', 'won_date']].copy()
deals_dates['won_date'] = pd.to_datetime(deals_dates['won_date'], dayfirst=True, errors='coerce')

date_temp = mql_dates.merge(deals_dates, on='mql_id', how='left')
master['days_to_convert'] = (date_temp['won_date'] - date_temp['first_contact_date']).dt.days

# Step 6: Time period columns
master['contact_month'] = master['first_contact_date'].dt.to_period('M')
master['won_month']     = pd.to_datetime(
    master['won_date'], dayfirst=True, errors='coerce'
).dt.to_period('M')

# Step 7: Fill revenue nulls for non-converted leads
master['total_revenue']    = master['total_revenue'].fillna(0)
master['total_orders']     = master['total_orders'].fillna(0)
master['total_items_sold'] = master['total_items_sold'].fillna(0)

# Save
master.to_csv(os.path.join(DATA_PATH, 'master_table.csv'), index=False)

print(f"Master table rebuilt: {master.shape[0]:,} rows x {master.shape[1]} columns")
print(f"\nConverted sellers  : {master['converted'].sum():,}")
print(f"Not converted      : {(master['converted']==0).sum():,}")
print(f"Conversion rate    : {round(master['converted'].mean()*100, 2)}%")
print(f"\nAvg days to convert (fixed): {master[master['converted']==1]['days_to_convert'].mean():.0f} days")
print(f"Avg review score            : {master['avg_review_score'].mean():.2f} / 5.0")
print(f"Avg revenue per seller      : ${master[master['total_revenue']>0]['total_revenue'].mean():,.2f}")
print(f"\nmaster_table.csv saved to data folder")

Rebuilding master table with clean data...



Master table rebuilt: 8,000 rows x 30 columns

Converted sellers  : 842
Not converted      : 7,158
Conversion rate    : 10.52%



Avg days to convert (fixed): -51 days
Avg review score            : 4.27 / 5.0
Avg revenue per seller      : $1,781.19

master_table.csv saved to data folder


In [13]:
# Look at the raw won_date values before any parsing
deals_raw = pd.read_csv(os.path.join(DATA_PATH, 'olist_closed_deals_dataset.csv'))

print("Raw won_date samples:")
print(deals_raw['won_date'].head(10).to_string())

print(f"\nfirst_contact_date sample from MQL:")
print(mql[mql['first_contact_date'].notna()]['first_contact_date'].head(5).to_string())

print(f"\nwon_date dtype in deals : {deals['won_date'].dtype}")
print(f"first_contact_date dtype: {mql['first_contact_date'].dtype}")

# Check a known converted lead end to end
sample = master[master['converted']==1][['mql_id','first_contact_date','won_date','days_to_convert']].head(5)
print(f"\nSample converted rows:")
print(sample.to_string())

Raw won_date samples:
0    2018-02-26 19:58:54
1    2018-05-08 20:17:59
2    2018-06-05 17:27:23
3    2018-01-17 13:51:03
4    2018-07-03 20:17:45
5    2018-02-07 18:04:05
6    2018-04-16 18:18:22
7    2018-04-17 17:01:57
8    2018-05-14 18:37:15
9    2018-04-24 03:00:00

first_contact_date sample from MQL:
0    2018-01-02
8    2017-10-11
12   2018-04-04
14   2018-03-04
15   2017-08-12

won_date dtype in deals : datetime64[ns]
first_contact_date dtype: datetime64[ns]

Sample converted rows:
                              mql_id first_contact_date            won_date  days_to_convert
4   5420aad7fec3549a85876ba1c529bd84                NaT 2018-02-26 19:58:54              NaN
12  a555fb36b9368110ede0f043dfc3b9a0         2018-04-04 2018-05-08 20:17:59            34.00
14  327174d3648a2d047e8940d7d15204ca         2018-03-04 2018-06-05 17:27:23            93.00
39  f5fee8f7da74f4887f5bcae2bafb6dd6                NaT 2018-01-17 13:51:03              NaN
67  ffe640179b554e295c167a2f6be528e0   

In [14]:
# Filter to converted leads where both dates exist
converted_with_dates = master[
    (master['converted'] == 1) &
    (master['days_to_convert'].notna()) &
    (master['days_to_convert'] > 0)
]

print("MASTER TABLE — FINAL SUMMARY")
print("=" * 55)
print(f"\n  Total leads                  : {len(master):,}")
print(f"  Converted sellers            : {master['converted'].sum():,}")
print(f"  Not converted                : {(master['converted']==0).sum():,}")
print(f"  Conversion rate              : {round(master['converted'].mean()*100, 2)}%")

print(f"\n  Leads WITH contact date      : {master['first_contact_date'].notna().sum():,}")
print(f"  Leads WITHOUT contact date   : {master['first_contact_date'].isna().sum():,}")

print(f"\n  Days to convert (where known): {converted_with_dates['days_to_convert'].mean():.0f} days avg")
print(f"  Fastest conversion           : {converted_with_dates['days_to_convert'].min():.0f} days")
print(f"  Slowest conversion           : {converted_with_dates['days_to_convert'].max():.0f} days")

print(f"\n  Sellers with revenue data    : {(master['total_revenue']>0).sum():,}")
print(f"  Avg revenue per seller       : ${master[master['total_revenue']>0]['total_revenue'].mean():,.2f}")
print(f"  Median revenue per seller    : ${master[master['total_revenue']>0]['total_revenue'].median():,.2f}")
print(f"  Total platform revenue       : ${master['total_revenue'].sum():,.2f}")
print(f"  Avg review score             : {master['avg_review_score'].mean():.2f} / 5.0")

print(f"\n  Channels breakdown:")
for ch, count in master['origin'].value_counts().items():
    converted_ch = master[(master['origin']==ch) & (master['converted']==1)].shape[0]
    conv_rate    = round(converted_ch / count * 100, 1)
    print(f"    {ch:<25} {count:>5,} leads   {converted_ch:>4} converted   ({conv_rate}% CVR)")

print(f"\n  Top 5 business segments (converted sellers):")
seg = master[master['converted']==1]['business_segment'].value_counts().head(5)
for segment, count in seg.items():
    print(f"    {segment:<35} {count:>4} sellers")

print(f"\nStage 1 complete — master_table.csv is your single source of truth")

MASTER TABLE — FINAL SUMMARY

  Total leads                  : 8,000
  Converted sellers            : 842
  Not converted                : 7,158
  Conversion rate              : 10.52%

  Leads WITH contact date      : 3,058
  Leads WITHOUT contact date   : 4,942

  Days to convert (where known): 109 days avg
  Fastest conversion           : 1 days
  Slowest conversion           : 588 days

  Sellers with revenue data    : 380
  Avg revenue per seller       : $1,781.19
  Median revenue per seller    : $547.40
  Total platform revenue       : $676,851.48
  Avg review score             : 4.27 / 5.0

  Channels breakdown:
    organic_search            2,296 leads    271 converted   (11.8% CVR)
    paid_search               1,586 leads    195 converted   (12.3% CVR)
    social                    1,350 leads     75 converted   (5.6% CVR)
    unknown                   1,159 leads    193 converted   (16.7% CVR)
    direct_traffic              499 leads     56 converted   (11.2% CVR)
    email